# Generic permitting history: Bronze
Upload the synthetic sample CSV to `Files/bronze/permits/history/` in the attached lakehouse first. All fields are synthetic. This notebook rebuilds the complete Bronze snapshot, so replay does not append duplicate files. Do not use this full-rebuild pattern for an unbounded production feed.


In [ ]:
history_path = 'Files/bronze/permits/history/*.csv'
bronze_table = 'dbo.bronze_permits'


In [ ]:
from pyspark.sql import functions as F, types as T

spark.conf.set('spark.sql.session.timeZone', 'UTC')
schema = T.StructType([
    T.StructField('permit_id', T.StringType(), False),
    T.StructField('status', T.StringType(), True),
    T.StructField('updated_at', T.TimestampType(), False),
    T.StructField('latitude', T.DoubleType(), True),
    T.StructField('longitude', T.DoubleType(), True),
])
history = (spark.read.option('header', True).option('mode', 'FAILFAST')
    .schema(schema).csv(history_path)
    .withColumn('_source_file', F.input_file_name())
    .withColumn('_load_ts', F.current_timestamp()))
if history.isEmpty():
    raise ValueError('No history rows found; refusing to replace the Bronze snapshot.')
if history.filter(F.col('permit_id').isNull() | (F.trim('permit_id') == '') | F.col('updated_at').isNull()).limit(1).count():
    raise ValueError('A permit business key or event timestamp is missing.')
# A complete snapshot overwrite makes repeated demo history loads replay-safe.
history.write.format('delta').mode('overwrite').saveAsTable(bronze_table)
display(spark.table(bronze_table))
